In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

In [2]:
df_protein = pd.read_excel('../mRNA_seq/mRNA_seq_AS_Control_gene_protein.xlsx')
#Eliminamos todas las variables que Norm ya que prefiero hacer nuestra propia normalizacion
df_protein = df_protein.loc[:, ~df_protein.columns.str.contains('Norm')]
df_protein = df_protein.set_index('hgnc_symbol').drop(['description','gene_biotype','chromosome_name'],axis = 1).T
#trasponemos para que cada fila sea un paciente y cada columna sea una variable , incluimos el simbolo de hgnc como nombre de columnas  y eliminamos variables que son string
df_protein = df_protein[['ACTA1', 'TNNT1', 'CYP2J2', 'HSPB6', 'MYL2', 'LTBP2', 'XPR1', 'SORT1',
       'PACS1', 'MFGE8', 'FSTL3', 'FGF12']]
df_noCoding = pd.read_excel('../mRNA_seq/mRNA_seq_AS_Control_LncRNA_identificados.xlsx')
df_noCoding = df_noCoding.loc[:, ~df_noCoding.columns.str.contains('Norm')]
#Eliminamos todas las variables que Norm ya que prefiero hacer nuestra propia normalizacion
df_noCoding = df_noCoding.set_index('hgnc_symbol').drop(['description','gene_biotype','chromosome_name'],axis = 1).T
df_noCoding = df_noCoding[['MBNL1-AS1', 'LINC01278', 'MIR1-1HG-AS1', 'LINC02208', 'LINC00702',
       'TNRC6C-AS1', 'H19']]

### Correlations

In [115]:
cor_matrix = pd.DataFrame(index=df_protein.columns, columns=df_noCoding.columns)
for gene in df_protein.columns:
    for lnc in df_noCoding.columns:
        rho, _ = spearmanr(df_protein[gene], df_noCoding[lnc])
        cor_matrix.loc[gene, lnc] = rho
corr_matrix = cor_matrix.reset_index().rename(columns = {'hgnc_symbol' : 'lncRNA'})
corr_matrix['Grupo'] = [1,1,1,1,1,2,2,2,2,2,2,3]

In [116]:
df_protein2 = df_protein.copy().reset_index()
df_protein2['Grupo'] = np.where(df_protein2['index'].str.contains('A'), 'Tratamiento', 
            np.where(df_protein2['index'].str.contains('C'), 'Control', 'Otro'))
df_protein2 = df_protein2.set_index('index')
df_protein2 = pd.get_dummies(df_protein2, columns=['Grupo'], drop_first=True)
df_protein2['Grupo_Tratamiento'] = df_protein2['Grupo_Tratamiento'].astype(int)
grupo_control = df_protein2[df_protein2['Grupo_Tratamiento'] == 0].drop(columns='Grupo_Tratamiento')
grupo_tratamiento = df_protein2[df_protein2['Grupo_Tratamiento'] == 1].drop(columns='Grupo_Tratamiento')
mean_control = grupo_control.mean()
mean_tratamiento = grupo_tratamiento.mean()
logFC = np.log2(mean_tratamiento / mean_control)
logFC_df = pd.DataFrame({'log2FC': logFC})
logFC_df = logFC_df.reset_index().rename(columns = {'index' : 'lncRNA'})
corr_matrix = pd.merge(corr_matrix,logFC_df, on = 'lncRNA', how = 'inner')
corr_matrix['Up/Down'] =  np.where(corr_matrix['log2FC'] > 0, 'Up', 'Down')
corr_matrix = corr_matrix[['lncRNA','Grupo','log2FC','Up/Down','MBNL1-AS1', 'LINC01278', 'MIR1-1HG-AS1', 'LINC02208',
       'LINC00702', 'TNRC6C-AS1', 'H19']]

In [117]:
corr_matrix.to_csv('Tabla_corr_variablesRepresentativas.csv',index = False)

Lo hacemos con todas las variables, las 78, no solo las representativas

In [118]:
df_protRelTotal = pd.read_csv('df_proteines_totalRelevant.csv',index_col = 0)
df_protcluster = pd.read_csv('vars_cluster.csv')

In [119]:
cor_matrix = pd.DataFrame(index=df_protRelTotal.columns, columns=df_noCoding.columns)
for gene in df_protRelTotal.columns:
    for lnc in df_noCoding.columns:
        rho, _ = spearmanr(df_protRelTotal[gene], df_noCoding[lnc])
        cor_matrix.loc[gene, lnc] = rho
corr_matrix = cor_matrix.reset_index().rename(columns = {'hgnc_symbol' : 'lncRNA'})
corr_matrix = corr_matrix.rename(columns = {'index' : 'variable'})
corr_matrix = corr_matrix.merge(df_protcluster, on = 'variable', how = 'inner').sort_values('grupo').reset_index(drop = True).rename(columns = {'variable' : 'lncRNA'})

In [120]:
df_protein2 = df_protRelTotal.copy().reset_index()
df_protein2['Grupo'] = np.where(df_protein2['index'].str.contains('A'), 'Tratamiento', 
            np.where(df_protein2['index'].str.contains('C'), 'Control', 'Otro'))
df_protein2 = df_protein2.set_index('index')
df_protein2 = pd.get_dummies(df_protein2, columns=['Grupo'], drop_first=True)
df_protein2['Grupo_Tratamiento'] = df_protein2['Grupo_Tratamiento'].astype(int)
grupo_control = df_protein2[df_protein2['Grupo_Tratamiento'] == 0].drop(columns='Grupo_Tratamiento')
grupo_tratamiento = df_protein2[df_protein2['Grupo_Tratamiento'] == 1].drop(columns='Grupo_Tratamiento')
mean_control = grupo_control.mean()
mean_tratamiento = grupo_tratamiento.mean()
logFC = np.log2(mean_tratamiento / mean_control)
logFC_df = pd.DataFrame({'log2FC': logFC})
logFC_df = logFC_df.reset_index().rename(columns = {'index' : 'lncRNA'})
corr_matrix = pd.merge(corr_matrix,logFC_df, on = 'lncRNA', how = 'inner')
corr_matrix['Up/Down'] =  np.where(corr_matrix['log2FC'] > 0, 'Up', 'Down')
corr_matrix = corr_matrix[['lncRNA','grupo','log2FC','Up/Down','MBNL1-AS1', 'LINC01278', 'MIR1-1HG-AS1', 'LINC02208',
       'LINC00702', 'TNRC6C-AS1', 'H19']].sort_values('grupo')


In [121]:
corr_matrix.to_csv('Tabla_corr_Todas_Variables.csv',index = False)

### ML

In [66]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics  import r2_score

In [9]:
lncRNA = df_noCoding
mRNA = df_protein
del df_noCoding,df_protein

In [74]:
def scl(df):
    scaler = StandardScaler()
    df_scl = pd.DataFrame(scaler.fit_transform(df), columns = df.columns, index=df.index)
    return df_scl
def pca90(df, prefix):
    scaler = StandardScaler()
    df_scl = scaler.fit_transform(df)
    pca = PCA(n_components=0.9)

    df_scl_comp = pca.fit_transform(df_scl)
    col_names = [f"{prefix}_comp{i+1}" for i in range(df_scl_comp.shape[1])]
    return pd.DataFrame(df_scl_comp, index = df.index, columns = col_names)
def discretizar_variables(df, q = 3):
    """
    Discretiza las variables continuas usando una cantidad definida de intervalos.
    """
    df_discretized = df.copy()
    for col in df.columns:
        df_discretized[col] = pd.qcut(df[col], q=q, labels=False, duplicates='drop')
    data_encoded = pd.DataFrame()
    for col in df_discretized.columns:
        dummies = pd.get_dummies(df_discretized[col], prefix=col, prefix_sep='_')
        data_encoded = pd.concat([data_encoded, dummies], axis=1)
    data_encoded = data_encoded.astype(int)

    return data_encoded
def model_reg (X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_train_pred = lr.predict(X_train)
    y_test_pred = lr.predict(X_test)

    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    return r2_train, r2_test

def model_rf(X,y,n_estimators = 100,max_depth = 2):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, oob_score=True, random_state=42)
    rf.fit(X_train, y_train)
    y_train_pred = rf.predict(X_train)
    y_test_pred = rf.predict(X_test)

    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    return r2_train, r2_test

def regression_expl(df_x, df_y, model):
    r2_set = {}
    for col_y in df_y.columns:
        y = df_y[col_y]
        X = df_x
        r2_train, r2_test = model(X, y)
        r2_set.update({col_y : {'r2_train':r2_train , 'r2_test':r2_test,'r2_diff' : abs(r2_train - r2_test)}})
    return pd.DataFrame(r2_set)


Con PCA previo

In [75]:
lncRNA_pca = pca90(lncRNA, 'lncRNA')
mRNA_scl = scl(mRNA)
regression_expl(df_x = lncRNA_pca, df_y = mRNA_scl, model = model_reg)


,ACTA1,TNNT1,CYP2J2,HSPB6,MYL2,LTBP2,XPR1,SORT1,PACS1,MFGE8,FSTL3,FGF12
r2_train,0.744996,0.635573,0.813716,0.619241,0.640089,0.737236,0.900160,0.911406,0.794515,0.862705,0.871632,0.342992
r2_test,0.747298,0.483641,0.607433,0.755323,0.318688,0.537884,0.883175,0.908652,0.757018,0.709893,0.813080,0.485579
r2_diff,0.002302,0.151932,0.206283,0.136083,0.321401,0.199352,0.016986,0.002754,0.037497,0.152812,0.058552,0.142587


In [76]:
regression_expl(df_x = lncRNA_pca, df_y = mRNA_scl, model = model_rf)

,ACTA1,TNNT1,CYP2J2,HSPB6,MYL2,LTBP2,XPR1,SORT1,PACS1,MFGE8,FSTL3,FGF12
r2_train,0.789414,0.809474,0.856293,0.812350,0.816374,0.856381,0.901272,0.882394,0.867400,0.922413,0.841612,0.564137
r2_test,0.713966,0.416474,0.633371,0.484198,0.309924,0.381936,0.659457,0.575635,0.737745,0.225165,0.781027,0.206109
r2_diff,0.075448,0.393000,0.222921,0.328152,0.506450,0.474446,0.241815,0.306759,0.129656,0.697248,0.060585,0.358027


### Variables Instrumentales

In [90]:
import statsmodels.api as sm

In [252]:
y = mRNA.copy().reset_index()
y['Grupo'] = np.where(y['index'].str.contains('A'), 'Tratamiento', 
              np.where(y['index'].str.contains('C'), 'Control', 'Otro'))
y = y.set_index('index')
y = pd.get_dummies(y, columns=['Grupo'], drop_first=True)
y['Grupo_Tratamiento'] = y['Grupo_Tratamiento'].astype(int)
y = y['Grupo_Tratamiento']
X = scl(mRNA.copy())
Z = scl(lncRNA.copy())

In [322]:
def get_vars_significativas(model):
    pvals = model.pvalues
    significant_vars_i = pvals[pvals < 0.1].index.tolist()
    return significant_vars_i
def var_endogena(y,X,Z):
    vars_endogenas = []
    modelos = []
    Z_const_new_list = []
    for x in X.columns:
        Z_const = sm.add_constant(Z)
        model2 = sm.OLS(X[x], Z_const).fit()
        Z_const_new = Z_const[get_vars_significativas(model2)]
        model2 = sm.OLS(X[x], Z_const_new).fit()
        X_hat = model2.fittedvalues
        X_new = pd.concat([X_hat, X.drop(x, axis = 1)], axis = 1).rename(columns = {0 : f'{x}_hat'})
        X_new_const = sm.add_constant(X_new)
        model3 = sm.OLS(y, X_new_const).fit()
        X_significativas = X_new_const[get_vars_significativas(model3)]
        if f'{x}_hat' in X_significativas:
            #print(x, 'es variable endógena')
            vars_endogenas.append(x)
            modelos.append(model3)
            Z_const_new_list.append(Z_const_new)
        #else:
            #print(x, 'NO es variable endógena')
    return modelos, vars_endogenas,Z_const_new_list
def instrumentos_validos(y,X,Z):
    from scipy.stats import chi2
    instr_validos = {}
    modelos, vars_endogenas,Z_const_new_list = var_endogena(y,X,Z)
    for var_endog in range(len(vars_endogenas)):
        model3 = modelos[var_endog]
        residuos3 = model3.resid
        Z_const_new = Z_const_new_list[var_endog]
        sargan_test  = sm.OLS(residuos3, Z_const_new).fit()
        sargan_test.summary()
        r2_sargan = sargan_test.rsquared
        n = len(y)
        sargan_stat = n * r2_sargan
        df_sargan = Z_const_new.shape[1] - 1 # sobreidentificación

        p_value = 1 - chi2.cdf(sargan_stat, df_sargan)
        if p_value > 0.1:
            print(vars_endogenas[var_endog], 'causa Y a través de instrumentos válidos y exógenos:', list(Z_const_new.columns))
            instr_validos.update({vars_endogenas[var_endog]: list(Z_const_new.columns)})
        else:
            print(vars_endogenas[var_endog],'no tiene instrumentos válidos o exógenos')
    return instr_validos

        

In [323]:
instrumentos_validos(y,X,Z)

TNNT1 causa Y a través de instrumentos válidos y exógenos: ['MBNL1-AS1', 'TNRC6C-AS1']
FGF12 causa Y a través de instrumentos válidos y exógenos: ['MIR1-1HG-AS1', 'LINC02208']


{'TNNT1': ['MBNL1-AS1', 'TNRC6C-AS1'], 'FGF12': ['MIR1-1HG-AS1', 'LINC02208']}

In [286]:
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      Grupo_Tratamiento   R-squared:                       0.754
Model:                            OLS   Adj. R-squared:                  0.619
Method:                 Least Squares   F-statistic:                     5.610
Date:                Wed, 25 Jun 2025   Prob (F-statistic):           0.000248
Time:                        16:55:03   Log-Likelihood:                 1.7148
No. Observations:                  35   AIC:                             22.57
Df Residuals:                      22   BIC:                             42.79
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6857      0.049     13.959      0.000       0.584       0.788
ACTA1          0.0111      0.156      0.071      0.944      -0.313       0.335
TNNT1         -0.1845      0.124     -1.487      0.151      -0.442       0.073
CYP2J2         0.0524      0.150      0.350      0.730      -0.258       0.363
HSPB6         -0.0669      0.141     -0.475      0.639      -0.359       0.225
MYL2           0.4984      0.168      2.963      0.007       0.150       0.847
LTBP2          0.2726      0.202      1.352      0.190      -0.146       0.691
XPR1          -0.0864      0.209     -0.413      0.684      -0.520       0.347
SORT1          0.0029      0.224      0.013      0.990      -0.462       0.468
PACS1          0.0636      0.160      0.398      0.695      -0.268       0.395
MFGE8         -0.1412      0.246     -0.574      0.572      -0.652       0.369
FSTL3         -0.1335      0.170     -0.784      0.441      -0.487       0.220
FGF12         -0.2848      0.085     -3.336      0.003      -0.462      -0.108
==============================================================================
Omnibus:                        0.109   Durbin-Watson:                   1.642
Prob(Omnibus):                  0.947   Jarque-Bera (JB):                0.218
Skew:                           0.119   Prob(JB):                        0.897
Kurtosis:                       2.695   Cond. No.                         18.7
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Estudiamos la endogeneidad de ACTA1

In [272]:
X_const = sm.add_constant(X[['ACTA1','MYL2','FGF12']])
model = sm.OLS(y, X_const).fit()
model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      Grupo_Tratamiento   R-squared:                       0.655
Model:                            OLS   Adj. R-squared:                  0.622
Method:                 Least Squares   F-statistic:                     19.62
Date:                Wed, 25 Jun 2025   Prob (F-statistic):           2.56e-07
Time:                        13:12:01   Log-Likelihood:                -4.1796
No. Observations:                  35   AIC:                             16.36
Df Residuals:                      31   BIC:                             22.58
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6857      0.049     14.002      0.000       0.586       0.786
ACTA1         -0.0517      0.096     -0.539      0.594      -0.247       0.144
MYL2           0.3012      0.089      3.397      0.002       0.120       0.482
FGF12         -0.2634      0.058     -4.532      0.000      -0.382      -0.145
==============================================================================
Omnibus:                        1.344   Durbin-Watson:                   1.125
Prob(Omnibus):                  0.511   Jarque-Bera (JB):                1.049
Skew:                           0.168   Prob(JB):                        0.592
Kurtosis:                       2.222   Cond. No.                         3.65
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [275]:
Z_const = sm.add_constant(Z[['LINC00702','TNRC6C-AS1','H19']])
model2 = sm.OLS(X['ACTA1'], Z[['LINC00702','TNRC6C-AS1','H19']]).fit()
model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                  ACTA1   R-squared (uncentered):                   0.764
Model:                            OLS   Adj. R-squared (uncentered):              0.742
Method:                 Least Squares   F-statistic:                              34.51
Date:                Wed, 25 Jun 2025   Prob (F-statistic):                    3.81e-10
Time:                        13:12:34   Log-Likelihood:                         -24.404
No. Observations:                  35   AIC:                                      54.81
Df Residuals:                      32   BIC:                                      59.47
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
LINC00702      0.4055      0.102      3.958      0.000       0.197       0.614
TNRC6C-AS1     0.3208      0.119      2.696      0.011       0.078       0.563
H19            0.3833      0.103      3.718      0.001       0.173       0.593
==============================================================================
Omnibus:                        1.027   Durbin-Watson:                   1.615
Prob(Omnibus):                  0.598   Jarque-Bera (JB):                0.327
Skew:                          -0.183   Prob(JB):                        0.849
Kurtosis:                       3.301   Cond. No.                         2.36
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Los instrumentos 'MYL2','FGF12' son relevantes. ahora veremos si son exógenos

In [276]:
ACTA1_hat = model2.fittedvalues
X_new = pd.concat([ACTA1_hat, X[['MYL2','FGF12']]], axis = 1).rename(columns = {0 : 'ACTA1_hat'})
X_new_const = sm.add_constant(X_new)
model3 = sm.OLS(y, X_new_const).fit()
model3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      Grupo_Tratamiento   R-squared:                       0.652
Model:                            OLS   Adj. R-squared:                  0.618
Method:                 Least Squares   F-statistic:                     19.36
Date:                Wed, 25 Jun 2025   Prob (F-statistic):           2.93e-07
Time:                        13:12:37   Log-Likelihood:                -4.3346
No. Observations:                  35   AIC:                             16.67
Df Residuals:                      31   BIC:                             22.89
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6857      0.049     13.940      0.000       0.585       0.786
ACTA1_hat     -0.0114      0.095     -0.119      0.906      -0.206       0.183
MYL2           0.2687      0.078      3.430      0.002       0.109       0.428
FGF12         -0.2496      0.055     -4.549      0.000      -0.361      -0.138
==============================================================================
Omnibus:                        2.361   Durbin-Watson:                   1.160
Prob(Omnibus):                  0.307   Jarque-Bera (JB):                1.487
Skew:                           0.240   Prob(JB):                        0.475
Kurtosis:                       2.112   Cond. No.                         3.12
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Comprobamos exogeneidad de los instrumentos estimando los instrumentos sobre los residuos del modelo anterior para verificar que los instrumentos solo afectan a Y a través de X

In [267]:
residuos3 = model3.resid
sargan_test  = sm.OLS(residuos3, Z_const).fit()
sargan_test.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.119
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     1.395
Date:                Wed, 25 Jun 2025   Prob (F-statistic):              0.263
Time:                        13:10:10   Log-Likelihood:                -2.1186
No. Observations:                  35   AIC:                             12.24
Df Residuals:                      31   BIC:                             18.46
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        2.88e-16      0.046   6.24e-15      1.000      -0.094       0.094
LINC00702      0.0679      0.055      1.234      0.227      -0.044       0.180
TNRC6C-AS1    -0.1261      0.064     -1.971      0.058      -0.257       0.004
H19            0.0748      0.055      1.349      0.187      -0.038       0.188
==============================================================================
Omnibus:                        1.257   Durbin-Watson:                   1.424
Prob(Omnibus):                  0.533   Jarque-Bera (JB):                1.063
Skew:                           0.214   Prob(JB):                        0.588
Kurtosis:                       2.262   Cond. No.                         2.36
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [268]:
from scipy.stats import chi2
r2_sargan = sargan_test.rsquared
n = len(y)
sargan_stat = n * r2_sargan
df_sargan = Z.shape[1] - 1 # sobreidentificación

p_value = 1 - chi2.cdf(sargan_stat, df_sargan)

print(f"Estadístico de Sargan: {sargan_stat:.4f}")
print(f"Grados de libertad: {df_sargan}")
print(f"P-valor: {p_value:.4f}")

Estadístico de Sargan: 4.1629
Grados de libertad: 6
P-valor: 0.6546


Son exógenos los instrumentos y válidos. Es decir, 'LINC00702','TNRC6C-AS1','H19' pueden afectar a ACTA1 que afecta a tener la enfermedad


Endogeneidad de LTBP2

In [288]:
Z_const = sm.add_constant(Z)
model2 = sm.OLS(X['LTBP2'], Z_const).fit()
model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  LTBP2   R-squared:                       0.755
Model:                            OLS   Adj. R-squared:                  0.691
Method:                 Least Squares   F-statistic:                     11.86
Date:                Wed, 25 Jun 2025   Prob (F-statistic):           8.29e-07
Time:                        17:06:28   Log-Likelihood:                -25.081
No. Observations:                  35   AIC:                             66.16
Df Residuals:                      27   BIC:                             78.60
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const         6.939e-17      0.095   7.28e-16      1.000      -0.196       0.196
MBNL1-AS1        0.5717      0.206      2.777      0.010       0.149       0.994
LINC01278       -0.0538      0.246     -0.219      0.829      -0.558       0.451
MIR1-1HG-AS1     0.0852      0.134      0.635      0.531      -0.190       0.360
LINC02208       -0.1715      0.105     -1.627      0.115      -0.388       0.045
LINC00702        0.1625      0.153      1.060      0.299      -0.152       0.477
TNRC6C-AS1       0.3080      0.135      2.289      0.030       0.032       0.584
H19             -0.1806      0.134     -1.347      0.189      -0.456       0.094
==============================================================================
Omnibus:                       14.288   Durbin-Watson:                   1.934
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               16.754
Skew:                           1.182   Prob(JB):                     0.000230
Kurtosis:                       5.429   Cond. No.                         6.27
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [289]:
LTBP2_hat = model2.fittedvalues
X_new = pd.concat([LTBP2_hat, X[['MYL2','FGF12']]], axis = 1).rename(columns = {0 : 'LTBP2_hat'})
X_new_const = sm.add_constant(X_new)
model3 = sm.OLS(y, X_new_const).fit()
model3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      Grupo_Tratamiento   R-squared:                       0.673
Model:                            OLS   Adj. R-squared:                  0.641
Method:                 Least Squares   F-statistic:                     21.28
Date:                Wed, 25 Jun 2025   Prob (F-statistic):           1.13e-07
Time:                        17:10:09   Log-Likelihood:                -3.2389
No. Observations:                  35   AIC:                             14.48
Df Residuals:                      31   BIC:                             20.70
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6857      0.048     14.384      0.000       0.588       0.783
LTBP2_hat      0.0928      0.065      1.421      0.165      -0.040       0.226
MYL2           0.2248      0.054      4.137      0.000       0.114       0.336
FGF12         -0.2258      0.050     -4.509      0.000      -0.328      -0.124
==============================================================================
Omnibus:                        2.592   Durbin-Watson:                   1.362
Prob(Omnibus):                  0.274   Jarque-Bera (JB):                1.713
Skew:                           0.318   Prob(JB):                        0.425
Kurtosis:                       2.122   Cond. No.                         1.89
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

# Borrador

In [ ]:
def interacciones_significativas (X,y):
    from itertools import combinations_with_replacement

    X_trans = pd.DataFrame()
    for i, j in combinations_with_replacement(X.columns, 2):
        X_trans[f'{i}_{j}'] = X[i] * X[j]
    lista_vars = list(range(0,len(X_trans.columns),10))
    lista_vars.append(len(X_trans.columns))
    significant_vars_list = []
    for i in range(len(lista_vars)-1):
        start = lista_vars[i]
        end = lista_vars[i + 1]
        X_const = sm.add_constant(X_trans[list(X_trans.columns[start:end])])
        model = sm.OLS(y, X_const).fit()
        model.summary()
        pvals = model.pvalues
        significant_vars_i = pvals[pvals < 0.1].index.tolist()
        significant_vars_list.append(significant_vars_i)
    significant_vars = list(set([j for i in significant_vars_list for j in i]))
    significant_vars.remove("const")
    return X_trans[significant_vars]